# Transformer(6.Cross Attention) 언어 모델 총정리 

## 1. Decoder Block
	◦ 트랜스포머 디코더 블록 단계별 공식
	1) 입력값(token) 
	2) 임베딩 + 위치인코딩 
	3) Masked Multi-Head Attention 
	4) 1차 Residual Connection → 1차 Layer Normalization
	5) Cross Attention(인코더 Key/Value, 디코더 Query 사용하여 Attention 계산) 
	6) 2차 Residual Connection → 2차 Layer Normalization 
	7) Feed Forward Network(FFN)
	8) 3차 Residual Connection → 3차 Layer Normalization

---
### 1) 임베딩 + 위치 인코딩
$ [ Y = \text{Embedding}(tokens) + \text{PositionalEncoding} ] $

### 2) Masked Multi-Head Attention
$ [ \text{MaskedMHA}(Y) = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right)V ] $

### 3) Residual Connection + Layer Normalization (1차)
$ [ H_1 = \text{LayerNorm}(Y + \text{MaskedMHA}(Y)) ] $

### 4) Cross Attention (인코더 Key/Value, 디코더 Query 사용하여 Attention 계산)
$ [ \text{CrossAttn}(H_1, H_{enc}) = \text{Softmax}\left(\frac{Q_{dec}K_{enc}^\top}{\sqrt{d_k}}\right)V_{enc} ] $

### 5) Residual Connection + Normalization (2차)
$ [ H_2 = \text{LayerNorm}(H_1 + \text{CrossAttn}(H_1, H_{enc})) ] $

### 6) Feed Forward Network (FFN)
$ [ \text{FFN}(H_2) = \max(0, H_2 W_1 + b_1) W_2 + b_2 ] $

### 7) Residual Connection + LayerNorm (3차)
$ [ H_3 = \text{LayerNorm}(H_2 + \text{FFN}(H_2)) ] $

---

## 2. Transformer(6.Cross Attention) 언어 모델의 Cross Attention 계산 과정을 공식과 함께 단계별로 계산
---
---
### 1) 입력값
1-1) 디코더 입력 (Residual + LayerNorm 결과):

$ [ H_{dec} = [-1.0, 1.0] ] $

1-2) 인코더 출력 (2차 Residual + LayerNorm 결과):

$ [ H_2 =
\begin{bmatrix}
1 & -1 \\
-1 & 1
\end{bmatrix} ] $

---
### 2) Query / Key / Value 생성
	◦ Attention 계산(Query/Key/Value 생성 단계)
$ [ Q = H_{dec} W_Q, \quad K = H_{2} W_K, \quad V = H_{2} W_V ] $

2-1) 예시 가중치 행렬(초기에 가중치는 램덤 설정된다) :

$ \text{WQ} = \begin{bmatrix}0.5 & -0.2 \\ 0.1 & 0.3\end{bmatrix} , $
$ \text{WK} = \begin{bmatrix}0.4 & 0.6 \\ -0.1 & 0.2\end{bmatrix} , $
$ \text{WV} = \begin{bmatrix}0.3 & 0.7 \\ 0.5 & -0.2\end{bmatrix} $

2-2) Query / Key / Value 생성(계산) 결과 :

Query 계산:
$ [ Q = H_{dec} W_Q = [-1.0, 1.0] \cdot
\begin{bmatrix}
0.5 & -0.2 \\
0.1 & 0.3
\end{bmatrix}
 ]
[ = [(-1)(0.5)+(1)(0.1), (-1)(-0.2)+(1)(0.3)]
 ]
[ = [-0.4, 0.5]
 ] $

Key 계산:
$ [ K = H_2 W_K =
\begin{bmatrix}
1 & -1 \\
-1 & 1
\end{bmatrix}
\cdot
\begin{bmatrix}
0.4 & 0.6 \\
-0.1 & 0.2
\end{bmatrix}
 ]
[ = \begin{bmatrix}
(1)(0.4)+(-1)(-0.1) & (1)(0.6)+(-1)(0.2) \\
(-1)(0.4)+(1)(-0.1) & (-1)(0.6)+(1)(0.2)
\end{bmatrix}
 ]
[ = \begin{bmatrix}
0.5 & 0.4 \\
-0.5 & -0.4
\end{bmatrix}
 ] $

Value 계산:
$ [ V = H_2 W_V =
\begin{bmatrix}
1 & -1 \\
-1 & 1
\end{bmatrix}
\cdot
\begin{bmatrix}
0.3 & 0.7 \\
0.5 & -0.2
\end{bmatrix}
 ]
[ = \begin{bmatrix}
(1)(0.3)+(-1)(0.5) & (1)(0.7)+(-1)(-0.2) \\
(-1)(0.3)+(1)(0.5) & (-1)(0.7)+(1)(-0.2)
\end{bmatrix}
 ]
[ = \begin{bmatrix}
-0.2 & 0.9 \\
0.2 & -0.9
\end{bmatrix}
 ] $

---
### 3) 벡터 간 유사도 계산 (내적)
	◦ Attention 계산(유사도 내적 계산 단계)
$ [ QK^\top ] $

Query와 Key의 내적을 통해 유사도를 계산합니다 : 

$ Key값을 전치행렬로 변환, [ K^T =\begin{bmatrix}
0.5 & -0.5 \\
0.4 & -0.4
\end{bmatrix}
 ] $

$ [ \text{Score} = Q K^T = [-0.4, 0.5] \cdot
\begin{bmatrix}
0.5 & -0.5 \\
0.4 & -0.4
\end{bmatrix}
 ] $

첫 번째 열 :

$ [ (-0.4)(0.5) + (0.5)(0.4) = -0.20 + 0.20 = 0 ] $

두 번째 열 :

$ [ (-0.4)(-0.5) + (0.5)(-0.4) = 0.20 - 0.20 = 0 ] $

따라서 :

$ [ \text{Score} = [0,0] ] $

---
### 4) 스케일링
	◦ Attention 계산(Scaled Dot-Product 단계)
$ [ \text{Scaled Score} = \frac{QK^\top}{\sqrt{d_k}} ] $

내적 $ QK^\top $ 값의 차원이 커질수록 커지게 되며, 값이 너무 커지면 Softmax에서 기울기 소실(Gradient Vanishing) 문재가 발생

따라서 $ [ \sigma = \sqrt{d_k} ] $ 로 나누어 값이 안정화되어 Softmax가 잘 작동됨, 
참고) Key 벡터 차원 $ [ d_k = 2 ] $

$ \text{Scaled Score} = \frac{QK^\top}{\sqrt{d_k}} $

$ = \frac{0}{\sqrt{2}} $

$ = \frac{0}{{1.414}} $

$ \approx 0 $

따라서 : 

$ [ z_1 = \frac{0}{1.414} = 0, \quad z_2 = \frac{0}{1.414} = 0 ] $

---
### 5) Softmax
	◦ 소트프맥스 기본 공식은 : 
$ [ P_i = \frac{e^{z_i}}{\sum_j e^{z_j}} ] $ 

$ [ z_i ] $ 특정 단어(토큰)에 대한 스케일된 Score, 
$ [ z_i = \frac{QK_i^\top}{\sqrt{d_k}} ] $
즉, Query와 해당 단어의 Key 벡터 내적을 스케일링한 값입니다.

$ [ z_j ] $ 모든 단어(토큰)에 대한 스케일된 Score 집합,
$ [ z_j = \frac{QK_j^\top}{\sqrt{d_k}}, \quad j = 1,2,\ldots,n ] $
즉, 현재 시점에서 볼 수 있는 모든 단어의 스코어를 의미합니다.

	◦ Attention 계산(Softmax 단계) 
$ [ \text{Score} = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} \right) ] $

따라서 Softmax 입력값은 :
앞서 스케일링한 값 
$ [ \frac{QK^\top}{\sqrt{d_k}} = 0 ] , $
$ [ z = [0, 0] ] $ 

Softmax 계산: $ 참고) [ e^{0} = 1 ] $

$ P1(\text{"첫번째행"}) = \frac{e^{1}}{e^{1} + e^{1}} = 0.5 $

$ P2(\text{"두번째행"}) = \frac{e^{1}}{e^{1} + e^{1}} = 0.5 $

---
### 6) Value 가중합
	◦ Attention 계산(Value 가중합 단계)
$ [ \text{CrossAttention}(Q,K,V) = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} \right)V ] $

Softmax 확률을 Value 벡터에 곱해 최종 Attention 출력을 얻습니다

$ \text{CrossAttention}(Q,K,V) $

$ = 0.5 \cdot [-0.2, 0.9] + 0.5 \cdot [0.2, -0.9] $

$ = [-0.1, 0.45] + [0.1, -0.45] = [0,0] $

---